# How to create a sensor

This tutorial demonstrates how to create a sensor.

There are different type of sensors available: irradiance sensor, radiance sensor, camera sensor.

## Prerequisites

### Perform imports

In [1]:
from pathlib import Path

In [2]:
from ansys.speos.core import Face, Project, Speos, launcher
from ansys.speos.core.generic.parameters import (
    CameraSensorParameters,
    ColorParameters,
    ObserverSensorParameters,
    PhotometricCameraParameters,
    PolarIntensityDimensionsParameters,
    PolarIntensityFormatTypes,
    PolarIntensitySensorParameters,
)
from ansys.speos.core.kernel.client import (
    default_docker_channel,
)
from ansys.speos.core.sensor import (
    Sensor3DIrradiance,
    SensorCamera,
    SensorImmersive,
    SensorIrradiance,
    SensorObserver,
    SensorPolarIntensity,
    SensorRadiance,
)

### Define constants

The constants help ensure consistency and avoid repetition throughout the example.

In [3]:
HOSTNAME = "localhost"
GRPC_PORT = 50098  # Be sure the Speos GRPC Server has been started on this port.
USE_DOCKER = True  # Set to False if you're running this example locally as a Notebook.
FILES = "CameraInputFiles"

### Define helper functions

In [4]:
def create_helper_geometries(project: Project):
    """Create bodies and faces."""

    def create_face(body):
        f = body.create_face(name="TheFaceF")
        f.vertices = [0, 0, 0, 1, 0, 0, 0, 1, 0]
        f.facets = [0, 1, 2]
        f.normals = [0, 0, 1, 0, 0, 1, 0, 0, 1]
        f.commit()

    root_part = project.create_root_part().commit()
    body_b1 = root_part.create_body(name="TheBodyB").commit()
    body_b2 = root_part.create_body(name="TheBodyC").commit()
    body_b3 = root_part.create_body(name="TheBodyD").commit()
    body_b4 = root_part.create_body(name="TheBodyE").commit()
    for b in [body_b1, body_b2, body_b3, body_b4]:
        create_face(b)

## Model Setup

### Load assets
The assets used to run this example are available in the
[PySpeos repository](https://github.com/ansys/pyspeos/) on GitHub.

> **Note:** Make sure you
> have downloaded simulation assets and set ``assets_data_path``
> to point to the assets folder.

In [5]:
if USE_DOCKER:  # Running on the remote server.
    assets_data_path = Path("/app") / "assets"
else:
    assets_data_path = Path("/path/to/your/download/assets/directory")

### Connect to the RPC Server
This Python client connects to a server where the Speos engine
is running as a service. In this example, the server and
client are the same machine. The launch_local_speos_rpc_method can
be used to start a local instance of the service.

In [6]:
if USE_DOCKER:
    speos = Speos(channel=default_docker_channel())
else:
    speos = launcher.launch_local_speos_rpc_server(port=GRPC_PORT)

/home/runner/work/pyspeos/pyspeos/.venv/lib/python3.14/site-packages/ansys/tools/common/cyberchannel.py:201: UserWarning: Starting gRPC client without TLS on localhost:50098. This is INSECURE. Consider using a secure connection.
  warn(f"Starting gRPC client without TLS on {target}. This is INSECURE. Consider using a secure connection.")


## Create a new project

The only way to create a sensor, is to create it from a project.

In [7]:
p = Project(speos=speos)
print(p)

{
    "name": "",
    "description": "",
    "metadata": {},
    "part_guid": "",
    "sub_scene_anchor_axis_system": [],
    "sources": [],
    "sensors": [],
    "simulations": [],
    "materials": [],
    "scenes": []
}


## Create

Create locally.

The mention "local: " is added when printing the sensor

In [8]:

distortion_file_path = str(assets_data_path / FILES / "CameraDistortion_130deg.OPTDistortion")
transmittance_file_path = str(assets_data_path / FILES / "CameraTransmittance.spectrum")
blue_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityBlue.spectrum")
green_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityGreen.spectrum")
red_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityRed.spectrum")

sensor1 = p.create_sensor(name="Camera.1", feature_type=SensorCamera)
sensor1.distortion_file_uri = distortion_file_path
# Choose photometric mode
sensor1.set_mode_photometric().transmittance_file_uri = transmittance_file_path
# Choose color mode (will imply to give spectrum file for blue, green and red)
mode_color = sensor1.photometric.set_mode_color()
mode_color.blue_spectrum_file_uri = blue_spectrum_path
mode_color.green_spectrum_file_uri = green_spectrum_path
mode_color.red_spectrum_file_uri = red_spectrum_path
print(sensor1)

local: {
    "name": "Camera.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "camera_sensor_template": {
            "sensor_mode_photometric": {
                "acquisition_integration": 0.01,
                "transmittance_file_uri": "/app/assets/CameraInputFiles/CameraTransmittance.spectrum",
                "gamma_correction": 2.2,
                "png_bits": "PNG_16",
                "color_mode_color": {
                    "red_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityRed.spectrum",
                    "green_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityGreen.spectrum",
                    "blue_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityBlue.spectrum",
                    "balance_mode_none": {}
                },
                "consider_diffraction_effects": false,
                "wavelengths_range": {
   

## Push it to the server.

Now that it is committed to the server, the mention "local: " is no more present when printing the
sensor.

In [9]:
sensor1.commit()
print(sensor1)

{
    "name": "Camera.1",
    "metadata": {
        "UniqueId": "981468ae-676e-43a8-8511-91ef2c4517b1"
    },
    "sensor_guid": "98962064-6ef1-4a45-8f37-fc709b22fa89",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "camera_sensor_template": {
            "sensor_mode_photometric": {
                "acquisition_integration": 0.01,
                "transmittance_file_uri": "/app/assets/CameraInputFiles/CameraTransmittance.spectrum",
                "gamma_correction": 2.2,
                "png_bits": "PNG_16",
                "color_mode_color": {
                    "red_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityRed.spectrum",
                    "green_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityGreen.spectrum",
                    "blue_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityBlue.spectrum",
                    "balance_mode_none": {}
                },
    

## Another example

Set more characteristics, and use parameters classes.

Camera feature is created with the same default values as the GUI speos.

If the user would like to modify the camera characteristics,
it is possible to do so as below.

In [10]:
distortion_file_path = str(assets_data_path / FILES / "CameraDistortion_130deg.OPTDistortion")
transmittance_file_path = str(assets_data_path / FILES / "CameraTransmittance.spectrum")
blue_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityBlue.spectrum")
green_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityGreen.spectrum")
red_spectrum_path = str(assets_data_path / FILES / "CameraSensitivityRed.spectrum")

color_params = ColorParameters(
    blue_spectrum_file_uri=blue_spectrum_path,
    green_spectrum_file_uri=green_spectrum_path,
    red_spectrum_file_uri=red_spectrum_path,
)
photo_params = PhotometricCameraParameters(
    color_mode=color_params, layer_type="by_source", transmittance_file_uri=transmittance_file_path
)
param = CameraSensorParameters(
    sensor_type_parameters=photo_params,
    distortion_file_uri=distortion_file_path,
    axis_system=[20, 10, 40, 1, 0, 0, 0, 1, 0, 0, 0, 1],
    # camera location [Origin, Xvector, Yvector, Zvector]
    focal_length=5.5,
    height=6,  # dimensions
    width=6,
)

sensor2 = p.create_sensor("Camera_Parameter", feature_type=SensorCamera, parameters=param)
sensor2.commit()

print(sensor2)

{
    "name": "Camera_Parameter",
    "metadata": {
        "UniqueId": "9412568a-ff66-4bf8-b5a5-4909d6fbba5a"
    },
    "sensor_guid": "4e649cb2-dce5-438a-886a-aae59d57ddbb",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "camera_sensor_template": {
            "sensor_mode_photometric": {
                "acquisition_integration": 0.01,
                "transmittance_file_uri": "/app/assets/CameraInputFiles/CameraTransmittance.spectrum",
                "gamma_correction": 2.2,
                "png_bits": "PNG_16",
                "color_mode_color": {
                    "red_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityRed.spectrum",
                    "green_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityGreen.spectrum",
                    "blue_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityBlue.spectrum",
                    "balance_mode_none": {}
               

## Read

### Sensor Instance

A mention "local: " is added if it is not yet committed to the server

In [11]:
print(sensor1)

{
    "name": "Camera.1",
    "metadata": {
        "UniqueId": "981468ae-676e-43a8-8511-91ef2c4517b1"
    },
    "sensor_guid": "98962064-6ef1-4a45-8f37-fc709b22fa89",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "camera_sensor_template": {
            "sensor_mode_photometric": {
                "acquisition_integration": 0.01,
                "transmittance_file_uri": "/app/assets/CameraInputFiles/CameraTransmittance.spectrum",
                "gamma_correction": 2.2,
                "png_bits": "PNG_16",
                "color_mode_color": {
                    "red_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityRed.spectrum",
                    "green_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityGreen.spectrum",
                    "blue_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityBlue.spectrum",
                    "balance_mode_none": {}
                },
    

### Project

Committed feature will appear in the project.

In [12]:
print(p)

{
    "sensors": [
        {
            "name": "Camera.1",
            "metadata": {
                "UniqueId": "981468ae-676e-43a8-8511-91ef2c4517b1"
            },
            "sensor_guid": "98962064-6ef1-4a45-8f37-fc709b22fa89",
            "display_name": "",
            "description": "",
            "result_file_name": "",
            "sensor": {
                "camera_sensor_template": {
                    "sensor_mode_photometric": {
                        "acquisition_integration": 0.01,
                        "transmittance_file_uri": "/app/assets/CameraInputFiles/CameraTransmittance.spectrum",
                        "gamma_correction": 2.2,
                        "png_bits": "PNG_16",
                        "color_mode_color": {
                            "red_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityRed.spectrum",
                            "green_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityGreen.spectrum",
     

## Update

Tipp: if you are manipulating a sensor already committed, don't forget to commit your changes.

If you don't, you will still only watch what is committed on the server.

In [13]:
# modify f number and axis system
sensor1.f_number = 11
sensor1.axis_system = [17, 10, 10, 1, 0, 0, 0, 1, 0, 0, 0, 1]
sensor1.commit()
print(sensor1)

{
    "name": "Camera.1",
    "metadata": {
        "UniqueId": "981468ae-676e-43a8-8511-91ef2c4517b1"
    },
    "sensor_guid": "98962064-6ef1-4a45-8f37-fc709b22fa89",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "camera_sensor_template": {
            "sensor_mode_photometric": {
                "acquisition_integration": 0.01,
                "transmittance_file_uri": "/app/assets/CameraInputFiles/CameraTransmittance.spectrum",
                "gamma_correction": 2.2,
                "png_bits": "PNG_16",
                "color_mode_color": {
                    "red_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityRed.spectrum",
                    "green_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityGreen.spectrum",
                    "blue_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityBlue.spectrum",
                    "balance_mode_none": {}
                },
    

In [14]:
# Modify photometric camera properties (including diffraction effects)
sensor1.distortion_file_uri = (
    assets_data_path / "CameraInputFiles" / "diffractive_effects.OPTDistortion"
)
sensor1.set_mode_photometric().set_mode_color().red_spectrum_file_uri = str(
    assets_data_path / "CameraInputFiles" / "CameraSensitivityRed.spectrum"
)
sensor1.set_mode_photometric().set_mode_color().green_spectrum_file_uri = str(
    assets_data_path / "CameraInputFiles" / "CameraSensitivityGreen.spectrum"
)
sensor1.set_mode_photometric().set_mode_color().blue_spectrum_file_uri = str(
    assets_data_path / "CameraInputFiles" / "CameraSensitivityBlue.spectrum"
)
wl = sensor1.set_mode_photometric().set_wavelengths_range()
wl.start = 486
wl.end = 655
wl.sampling = 13
sensor1.set_mode_photometric().consider_diffraction_effects = True
sensor1.commit()
print(sensor1)

{
    "name": "Camera.1",
    "metadata": {
        "UniqueId": "981468ae-676e-43a8-8511-91ef2c4517b1"
    },
    "sensor_guid": "98962064-6ef1-4a45-8f37-fc709b22fa89",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "camera_sensor_template": {
            "sensor_mode_photometric": {
                "acquisition_integration": 0.01,
                "gamma_correction": 2.2,
                "png_bits": "PNG_16",
                "color_mode_color": {
                    "red_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityRed.spectrum",
                    "green_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityGreen.spectrum",
                    "blue_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityBlue.spectrum",
                    "balance_mode_none": {}
                },
                "consider_diffraction_effects": true,
                "wavelengths_range": {
              

/tmp/ipykernel_15641/910970107.py:19: UserWarning: Please note that the following values ['focal_length', 'imager_distance', 'f_number', 'Transmittance Spectrum'] were over written by the values stored in the distortion file
  sensor1.commit()


## Reset

Possibility to reset local values from the one available in the server.

In [15]:
sensor1.set_mode_geometric()  # set camera in geometric mode but no commit
sensor1.reset()  # reset -> this will apply the server value (photometric mode) to the local value
sensor1.delete()  # delete (to display the local value with the below print)
print(sensor1._sensor_template)

ansys.api.speos.sensor.v1.SensorTemplate
{
    "camera_sensor_template": {
        "sensor_mode_photometric": {
            "acquisition_integration": 0.01,
            "gamma_correction": 2.2,
            "png_bits": "PNG_16",
            "color_mode_color": {
                "red_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityRed.spectrum",
                "green_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityGreen.spectrum",
                "blue_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityBlue.spectrum",
                "balance_mode_none": {}
            },
            "consider_diffraction_effects": true,
            "wavelengths_range": {
                "w_start": 486.0,
                "w_end": 655.0,
                "w_sampling": 13
            },
            "acquisition_lag_time": 0.0,
            "transmittance_file_uri": ""
        },
        "horz_pixel": 640,
        "vert_pixel": 480,
        "width": 5.0,


## Delete

Once the data is deleted from the server, you can still work with local data and maybe commit
later.

In [16]:
sensor2.delete()
print(sensor2)

local: {
    "name": "Camera_Parameter",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "camera_sensor_template": {
            "sensor_mode_photometric": {
                "acquisition_integration": 0.01,
                "transmittance_file_uri": "/app/assets/CameraInputFiles/CameraTransmittance.spectrum",
                "gamma_correction": 2.2,
                "png_bits": "PNG_16",
                "color_mode_color": {
                    "red_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityRed.spectrum",
                    "green_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityGreen.spectrum",
                    "blue_spectrum_file_uri": "/app/assets/CameraInputFiles/CameraSensitivityBlue.spectrum",
                    "balance_mode_none": {}
                },
                "consider_diffraction_effects": false,
                "wavelengths_range

In [17]:
sensor1.delete()

## Other sensors

### Irradiance sensor

In [18]:
sensor3 = p.create_sensor(name="Irradiance.1", feature_type=SensorIrradiance)
sensor3.commit()
print(sensor3)

{
    "name": "Irradiance.1",
    "metadata": {
        "UniqueId": "c752907d-56d6-4863-9a52-240b487dd28f"
    },
    "sensor_guid": "ddf61c7f-c86a-4f18-94f7-c964719cf399",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "irradiance_sensor_template": {
            "sensor_type_photometric": {},
            "illuminance_type_planar": {},
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ],
            "layer_type_none": {},
            "ray_file_type": "RayFileN

In [19]:
sensor3.set_type_radiometric()
sensor3.set_layer_type_polarization()
sensor3.commit()
print(sensor3)

{
    "name": "Irradiance.1",
    "metadata": {
        "UniqueId": "c752907d-56d6-4863-9a52-240b487dd28f"
    },
    "sensor_guid": "ddf61c7f-c86a-4f18-94f7-c964719cf399",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "irradiance_sensor_template": {
            "sensor_type_radiometric": {},
            "illuminance_type_planar": {},
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ],
            "layer_type_polarization": {},
            "ray_file_type": "

In [20]:
sensor3.delete()

### radiance sensor

In [21]:
sensor4 = p.create_sensor(name="Radiance.1", feature_type=SensorRadiance)
sensor4.commit()
print(sensor4)

{
    "name": "Radiance.1",
    "metadata": {
        "UniqueId": "9bd03529-ef12-4b7f-974c-4fc1cf4fc5cb"
    },
    "sensor_guid": "3da0385d-d29e-4a22-8f59-0c88181afc22",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "radiance_sensor_template": {
            "sensor_type_photometric": {},
            "focal": 250.0,
            "integration_angle": 5.0,
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ],
            "layer_type_none": {},
            "observ

In [22]:
sensor4.focal = 200
sensor4.set_type_spectral()
sensor4.set_layer_type_source()
sensor4.commit()
print(sensor4)

{
    "name": "Radiance.1",
    "metadata": {
        "UniqueId": "9bd03529-ef12-4b7f-974c-4fc1cf4fc5cb"
    },
    "sensor_guid": "3da0385d-d29e-4a22-8f59-0c88181afc22",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "radiance_sensor_template": {
            "sensor_type_spectral": {
                "wavelengths_range": {
                    "w_start": 400.0,
                    "w_end": 700.0,
                    "w_sampling": 13
                }
            },
            "focal": 200.0,
            "integration_angle": 5.0,
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                

In [23]:
sensor4.delete()
print(sensor4)

local: {
    "name": "Radiance.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "radiance_sensor_template": {
            "sensor_type_spectral": {
                "wavelengths_range": {
                    "w_start": 400.0,
                    "w_end": 700.0,
                    "w_sampling": 13
                }
            },
            "focal": 200.0,
            "integration_angle": 5.0,
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
    

### 3D irradiance sensor

In [24]:
create_helper_geometries(p)
sensor5 = p.create_sensor(name="3D_Irradiance.2", feature_type=Sensor3DIrradiance)
face = p.find(name="TheBodyB/TheFaceF", feature_type=Face)[0]
sensor5.geometries = [face]
sensor5.commit()
print(sensor5)

{
    "name": "3D_Irradiance.2",
    "metadata": {
        "UniqueId": "0a84b4b4-d37d-4a89-8b90-7bb9fc364232"
    },
    "sensor_guid": "53f96e58-6679-4988-a2b4-e9e25cca57ca",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "3D_Irradiance.2",
        "irradiance_3d": {
            "type_photometric": {
                "integration_type_planar": {
                    "reflection": true,
                    "transmission": true,
                    "absorption": true
                }
            },
            "layer_type_none": {},
            "geometries": {
                "geo_paths": [
                    "TheBodyB/TheFaceF"
                ]
            },
            "ray_file_type": "RayFileNone"
        },
        "description": "",
        "metadata": {}
    }
}


### Immersive sensor

An immersive sensor wraps the observer in a virtual cube and records light arriving from all
six directions (front, back, left, right, top, bottom).  It can be used with both direct and
inverse simulations.


Create an immersive sensor and commit it to show its default settings.

In [25]:
sensor6 = p.create_sensor(name="Immersive.1", feature_type=SensorImmersive)
print(sensor6)  # local: not yet on server

local: {
    "name": "Immersive.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Immersive.1",
        "immersive_sensor_template": {
            "sampling": 600,
            "integration_angle": 5.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "exclude_faces": {
                "front": false,
                "back": false,
                "left": false,
                "right": false,
                "top": false,
                "bottom": false
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ],
            "la

In [26]:
sensor6.commit()
print(sensor6)  # now on server

{
    "name": "Immersive.1",
    "metadata": {
        "UniqueId": "2d6159bb-c071-4086-95d2-4d919dc6ed76"
    },
    "sensor_guid": "02fb9e53-9c03-4dd2-a510-8fca145c4022",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Immersive.1",
        "immersive_sensor_template": {
            "sampling": 600,
            "integration_angle": 5.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "exclude_faces": {
                "front": false,
                "back": false,
                "left": false,
                "right": false,
                "top": false,
                "bottom": false
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,

**Customise the sensor**

The sampling controls the pixel count per cube face, the integration angle is used for
direct simulations, and the wavelengths range selects the spectral window.

In [27]:
sensor6.sampling = 256
sensor6.integration_angle = 10.0
sensor6.stereo_interocular_distance = 50

In [28]:
wl = sensor6.set_wavelengths_range()
wl.start = 380.0
wl.end = 780.0
wl.sampling = 20

In [29]:
# Exclude the bottom face so rays from below are not recorded.
sensor6.exclude_bottom = True

In [30]:
# Separate results by light source.
sensor6.set_layer_type_source()

In [31]:
# Reposition the sensor (Origin, X-axis, Y-axis, Z-axis).
sensor6.axis_system = [5, 0, 10, 1, 0, 0, 0, 1, 0, 0, 0, 1]

In [32]:
sensor6.commit()
print(sensor6)

{
    "name": "Immersive.1",
    "metadata": {
        "UniqueId": "2d6159bb-c071-4086-95d2-4d919dc6ed76"
    },
    "sensor_guid": "02fb9e53-9c03-4dd2-a510-8fca145c4022",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Immersive.1",
        "immersive_sensor_template": {
            "sampling": 256,
            "stereo": {
                "interocular_distance": 50.0
            },
            "integration_angle": 10.0,
            "wavelengths_range": {
                "w_start": 380.0,
                "w_end": 780.0,
                "w_sampling": 20
            },
            "exclude_faces": {
                "bottom": true,
                "front": false,
                "back": false,
                "left": false,
                "right": false,
                "top": false
            },
            "axis_system": [
                5.0,
                0.0,
                10.0,
                1.0,
                0.0

**Reset and delete**

In [33]:
sensor6.sampling = 512  # local modification — not yet committed
sensor6.reset()  # restores the last committed value (256)
print(sensor6)

{
    "name": "Immersive.1",
    "metadata": {
        "UniqueId": "2d6159bb-c071-4086-95d2-4d919dc6ed76"
    },
    "sensor_guid": "02fb9e53-9c03-4dd2-a510-8fca145c4022",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Immersive.1",
        "immersive_sensor_template": {
            "sampling": 256,
            "stereo": {
                "interocular_distance": 50.0
            },
            "integration_angle": 10.0,
            "wavelengths_range": {
                "w_start": 380.0,
                "w_end": 780.0,
                "w_sampling": 20
            },
            "exclude_faces": {
                "bottom": true,
                "front": false,
                "back": false,
                "left": false,
                "right": false,
                "top": false
            },
            "axis_system": [
                5.0,
                0.0,
                10.0,
                1.0,
                0.0

In [34]:
sensor6.delete()
print(sensor6)

local: {
    "name": "Immersive.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Immersive.1",
        "immersive_sensor_template": {
            "sampling": 256,
            "stereo": {
                "interocular_distance": 50.0
            },
            "integration_angle": 10.0,
            "wavelengths_range": {
                "w_start": 380.0,
                "w_end": 780.0,
                "w_sampling": 20
            },
            "exclude_faces": {
                "bottom": true,
                "front": false,
                "back": false,
                "left": false,
                "right": false,
                "top": false
            },
            "axis_system": [
                5.0,
                0.0,
                10.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
       

### Observer sensor

An observer sensor places multiple virtual viewpoints on a sphere around the scene.
It is useful when you want to sample a setup from several directions with one feature.

**Default values**

In [35]:
sensor7 = p.create_sensor(name="Observer.1", feature_type=SensorObserver)
print(sensor7)  # local: not yet on server

local: {
    "name": "Observer.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.1",
        "observer_sensor_template": {
            "focal": 50.0,
            "integration_angle": 5.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "distance": 1000.0,
            "sensors_locations": {
                "x_start": -45.0,
                "x_end": 45.0,
                "x_sampling": 5,
                "y_start": -30.0,
                "y_end": 30.0,
                "y_sampling": 3
            },
            "axis_system": [
 

In [36]:
sensor7.commit()
print(sensor7)  # now on server

{
    "name": "Observer.1",
    "metadata": {
        "UniqueId": "777140de-ff5f-4ce4-8837-e7c8d16d1363"
    },
    "sensor_guid": "ad0db749-3825-484f-81d2-ccb43f444364",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.1",
        "observer_sensor_template": {
            "focal": 50.0,
            "integration_angle": 5.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "distance": 1000.0,
            "sensors_locations": {
                "x_start": -45.0,
                "x_end": 45.0,
                "x_sampling": 5,
                "y_start": -30.0,
               

**Customise the sensor**

The focal distance defines the observer plane, ``distance`` controls the radius of the
sampling sphere, and the angular range controls how many viewpoints are created.

In [37]:
sensor7.focal = 320.0
sensor7.integration_angle = 7.5
sensor7.distance = 130.0
sensor7.stereo_interocular_distance = 63.0

In [38]:
observer_wl = sensor7.set_wavelengths_range()
observer_wl.start = 430.0
observer_wl.end = 670.0
observer_wl.sampling = 18

In [39]:
observer_dims = sensor7.set_dimensions()
observer_dims.x_start = -90.0
observer_dims.x_end = 90.0
observer_dims.x_sampling = 110
observer_dims.y_start = -70.0
observer_dims.y_end = 70.0
observer_dims.y_sampling = 75

In [40]:
observer_angles = sensor7.set_angular_range()
observer_angles.x_start = -55.0
observer_angles.x_end = 55.0
observer_angles.x_sampling = 9
observer_angles.y_start = -42.0
observer_angles.y_end = 42.0
observer_angles.y_sampling = 6

In [41]:
# Separate the recorded results by source and move the sensor frame.
sensor7.set_layer_type_source()
sensor7.axis_system = [10, 0, 15, 1, 0, 0, 0, 1, 0, 0, 0, 1]

In [42]:
sensor7.commit()
print(sensor7)

{
    "name": "Observer.1",
    "metadata": {
        "UniqueId": "777140de-ff5f-4ce4-8837-e7c8d16d1363"
    },
    "sensor_guid": "ad0db749-3825-484f-81d2-ccb43f444364",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.1",
        "observer_sensor_template": {
            "focal": 320.0,
            "integration_angle": 7.5,
            "wavelengths_range": {
                "w_start": 430.0,
                "w_end": 670.0,
                "w_sampling": 18
            },
            "dimensions": {
                "x_start": -90.0,
                "x_end": 90.0,
                "x_sampling": 110,
                "y_start": -70.0,
                "y_end": 70.0,
                "y_sampling": 75
            },
            "stereo": {
                "interocular_distance": 63.0
            },
            "distance": 130.0,
            "sensors_locations": {
                "x_start": -55.0,
                "x_end": 55.0,

**Create from parameter dataclass**

Observer sensors can also be configured in one shot with a parameter dataclass.

In [43]:
observer_params = ObserverSensorParameters(
    focal=280.0,
    integration_angle=6.0,
    distance=110.0,
    interocular_distance=66.0,
    axis_system=[5, 10, 15, 1, 0, 0, 0, 1, 0, 0, 0, 1],
    layer_type="by_source",
)
sensor8 = p.create_sensor(
    name="Observer.Parameters",
    feature_type=SensorObserver,
    parameters=observer_params,
)
sensor8.commit()
print(sensor8)

{
    "name": "Observer.Parameters",
    "metadata": {
        "UniqueId": "ec8ef05b-bb8c-46e0-a2ac-9568c204346d"
    },
    "sensor_guid": "e11a92e3-c0a7-40c0-b227-417fa38bbc7f",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.Parameters",
        "observer_sensor_template": {
            "focal": 280.0,
            "integration_angle": 6.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "stereo": {
                "interocular_distance": 66.0
            },
            "distance": 110.0,
            "sensors_locations": {
                "x_start": -45.0,
           

**Reset and delete**

In [44]:
sensor7.distance = 300.0  # local modification — not yet committed
sensor7.stereo_interocular_distance = None
sensor7.reset()  # restores the last committed values from the server
print(sensor7)

{
    "name": "Observer.1",
    "metadata": {
        "UniqueId": "777140de-ff5f-4ce4-8837-e7c8d16d1363"
    },
    "sensor_guid": "ad0db749-3825-484f-81d2-ccb43f444364",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.1",
        "observer_sensor_template": {
            "focal": 320.0,
            "integration_angle": 7.5,
            "wavelengths_range": {
                "w_start": 430.0,
                "w_end": 670.0,
                "w_sampling": 18
            },
            "dimensions": {
                "x_start": -90.0,
                "x_end": 90.0,
                "x_sampling": 110,
                "y_start": -70.0,
                "y_end": 70.0,
                "y_sampling": 75
            },
            "stereo": {
                "interocular_distance": 63.0
            },
            "distance": 130.0,
            "sensors_locations": {
                "x_start": -55.0,
                "x_end": 55.0,

In [45]:
sensor8.delete()
print(sensor8)

local: {
    "name": "Observer.Parameters",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.Parameters",
        "observer_sensor_template": {
            "focal": 280.0,
            "integration_angle": 6.0,
            "wavelengths_range": {
                "w_start": 400.0,
                "w_end": 700.0,
                "w_sampling": 13
            },
            "dimensions": {
                "x_start": -50.0,
                "x_end": 50.0,
                "x_sampling": 100,
                "y_start": -50.0,
                "y_end": 50.0,
                "y_sampling": 100
            },
            "stereo": {
                "interocular_distance": 66.0
            },
            "distance": 110.0,
            "sensors_locations": {
                "x_start": -45.0,
                "x_end": 45.0,
                "x_sampling": 5,
                "y_start": -30.0,
      

In [46]:
sensor7.delete()
print(sensor7)

local: {
    "name": "Observer.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "Observer.1",
        "observer_sensor_template": {
            "focal": 320.0,
            "integration_angle": 7.5,
            "wavelengths_range": {
                "w_start": 430.0,
                "w_end": 670.0,
                "w_sampling": 18
            },
            "dimensions": {
                "x_start": -90.0,
                "x_end": 90.0,
                "x_sampling": 110,
                "y_start": -70.0,
                "y_end": 70.0,
                "y_sampling": 75
            },
            "stereo": {
                "interocular_distance": 63.0
            },
            "distance": 130.0,
            "sensors_locations": {
                "x_start": -55.0,
                "x_end": 55.0,
                "x_sampling": 9,
                "y_start": -42.0,
                "y_end": 

### Polar intensity sensor

A polar intensity sensor generates an IES/Eulumdat photometric file from the simulation.
It supports format selection, explicit angular sampling, and far-field or near-field setup.

In [47]:
sensor_polar = p.create_sensor(name="PolarIntensity.1", feature_type=SensorPolarIntensity)
print(sensor_polar)  # local: not yet on server

local: {
    "name": "PolarIntensity.1",
    "display_name": "",
    "description": "",
    "metadata": {},
    "sensor_guid": "",
    "result_file_name": "",
    "sensor": {
        "name": "PolarIntensity.1",
        "polar_intensity_sensor_template": {
            "iesna_c": {},
            "dimensions": {
                "horizontal_sampling": 720,
                "vertical_sampling": 361
            },
            "far_field": {
                "integration_angle": 1.0
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "metadata": {}
    }
}


In [48]:
sensor_polar.commit()
print(sensor_polar)  # now on server

{
    "name": "PolarIntensity.1",
    "metadata": {
        "UniqueId": "b16ddcff-f269-4ba2-a552-27484af2f313"
    },
    "sensor_guid": "7af65e05-6ee4-471f-99b4-001796c6a43b",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "PolarIntensity.1",
        "polar_intensity_sensor_template": {
            "iesna_c": {},
            "dimensions": {
                "horizontal_sampling": 720,
                "vertical_sampling": 361
            },
            "far_field": {
                "integration_angle": 1.0
            },
            "axis_system": [
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "metadata": {}
    }
}


In [49]:
# Customise format, sampling and field configuration.
assert isinstance(sensor_polar, SensorPolarIntensity)
sensor_polar.set_format_eulumdat()
sensor_polar.set_constant_sampling()
sensor_polar.horizontal_sampling = 180
sensor_polar.vertical_sampling = 90
sensor_polar.set_far_field()
sensor_polar.integration_angle = 1
sensor_polar.axis_system = [0, 0, 25, 1, 0, 0, 0, 1, 0, 0, 0, 1]

In [50]:
sensor_polar.commit()
print(sensor_polar)

{
    "name": "PolarIntensity.1",
    "metadata": {
        "UniqueId": "b16ddcff-f269-4ba2-a552-27484af2f313"
    },
    "sensor_guid": "7af65e05-6ee4-471f-99b4-001796c6a43b",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "PolarIntensity.1",
        "polar_intensity_sensor_template": {
            "eulumdat": {},
            "dimensions": {
                "horizontal_sampling": 180,
                "vertical_sampling": 90
            },
            "far_field": {
                "integration_angle": 1.0
            },
            "axis_system": [
                0.0,
                0.0,
                25.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "metadata": {}
    }
}


In [51]:
# The same sensor can be created directly from parameter dataclasses.
polar_param = PolarIntensitySensorParameters(
    format=PolarIntensityFormatTypes.iesna_b,
    dimensions=PolarIntensityDimensionsParameters(horizontal_sampling=37, vertical_sampling=37),
    integration_angle=1.0,
    axis_system=[10, 0, 30, 1, 0, 0, 0, 1, 0, 0, 0, 1],
)

In [52]:
sensor_polar_param = p.create_sensor(
    name="PolarIntensity.Parameters",
    feature_type=SensorPolarIntensity,
    parameters=polar_param,
)
sensor_polar_param.commit()
print(sensor_polar_param)

{
    "name": "PolarIntensity.Parameters",
    "metadata": {
        "UniqueId": "e22f4dcc-38d4-4191-a769-28c30a2ab853"
    },
    "sensor_guid": "8b04484b-0f3c-4ac0-b7a3-dba8d8cd1c06",
    "display_name": "",
    "description": "",
    "result_file_name": "",
    "sensor": {
        "name": "PolarIntensity.Parameters",
        "polar_intensity_sensor_template": {
            "iesna_b": {},
            "dimensions": {
                "horizontal_sampling": 37,
                "vertical_sampling": 37
            },
            "far_field": {
                "integration_angle": 1.0
            },
            "axis_system": [
                10.0,
                0.0,
                30.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0
            ]
        },
        "description": "",
        "metadata": {}
    }
}


In [53]:
sensor_polar_param.delete()
sensor_polar.delete()

In [54]:
speos.close()

True